In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings 
from langchain_groq import ChatGroq
from langchain_core.prompts import  PromptTemplate
from langchain_chroma import Chroma

In [3]:
loader = PyPDFLoader("../data/resume1 (1).pdf")
docs = loader.load()
len(docs)

1

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200 )
splitted_data = splitter.split_documents(docs)
len(splitted_data)

5

In [5]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [6]:
vector_store = Chroma.from_documents(
    documents = splitted_data ,
    embedding=embeddings,
    
)

In [18]:
query = "What are this person's machine learning and data science skills and projects?"
data = vector_store.similarity_search(query=query)

In [27]:
context =""
for doc in data :
    context += doc.page_content + "/n"

In [21]:
llm = ChatGroq(model="openai/gpt-oss-20b")


In [ ]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data :
        context += doc.page_content + "\n"

    return {
        "context" : context ,
        "question":query
    }

    

In [26]:
prompt = PromptTemplate.from_template("""
  You are a helpful assiant  and provide answered based onthe context for user question .
  if you don't known the answer , then you can say that 'I dint know '
  Context : {context}
  Question: {question}
""")

In [28]:
rag_chain = get_context | prompt | llm

In [ ]:
res =  rag_chain.invoke("What is the duraton of m")